## Data Manipulation and Exploration with Spark

### Set up the Azure ML Client

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id="YOUR_SUBSCRIPTION_ID",
    resource_group_name="YOUR_RESOURCE_GROUP_NAME",
    workspace_name="YOUR_WORKSPACE_NAME"
)

### Get the Default Datastore

In [ ]:
# Get the default datastore
default_ds = ml_client.datastores.get_default()

# Enumerate all datastores and indicate which one is the default
for ds in ml_client.datastores.list():
    print(ds.name, "- Default =", ds.name == default_ds.name)

### Upload Data to the Datastore

In [ ]:
import requests
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

datastore = ml_client.datastores.get_default()

datastore_path = f"azureml://datastores/{datastore.name}/paths/sales-data/"

# Register folder of CSVs as a data asset
data_asset = Data(
    path=datastore_path,
    type=AssetTypes.URI_FOLDER,
    name="sales-data",
    description="Sales CSV files",
)

ml_client.data.create_or_update(data_asset)

### Create a Tabular Dataset from the Uploaded Data

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

df = spark.read.load(
    f"azureml://datastores/{datastore.name}/paths/sales-data/*.csv",
    format='csv',
)

display(df)

### Defining schema for the dataframe

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

df = spark.read.load(
    "azureml://datastores/aml_datastore/paths/sales-data/*.csv",
    format='csv',
    schema=orderSchema
)

display(df)

### Query Data using Spark SQL

In [ ]:
df.createOrReplaceTempView("salesorders")
spark_df = spark.sql("SELECT * FROM salesorders")
display(spark_df)

In [ ]:
sqlQuery = "SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
               SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
        FROM salesorders \
        GROUP BY CAST(YEAR(OrderDate) AS CHAR(4)) \
        ORDER BY OrderYear"
df_spark = spark.sql(sqlQuery)
df_spark.show()

### Using Matplotlib to visualize the data

In [ ]:
from matplotlib import pyplot as plt

# Convert Spark → Pandas
df_sales = df_spark.toPandas()

# Drop rows with null values
df_sales = df_sales.dropna(subset=["OrderYear", "GrossRevenue"])

# Plot
plt.bar(x=df_sales["OrderYear"], height=df_sales["GrossRevenue"])
plt.show()

### Using Seaborn to visualize the data

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

# Convert Spark → Pandas if not already done
df_sales = df_spark.toPandas()

# Remove rows with null values
df_sales = df_sales.dropna(subset=["OrderYear", "GrossRevenue"])

# Clear plot
plt.clf()

# Create bar chart
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_sales)

plt.show()